# einops-rearrange — ex3: image flatten (axis composition)

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `einops-rearrange`. When a test cell passes, your progress is reported back to your account.

**What you'll practice.** Five `einops.rearrange` patterns that ramp from identity → axis swap → composition → decomposition → patching. Read the docstring, fill the function body, run the test cell. The solution sits in the collapsed `<details>` block below each exercise.

**Per-exercise structure** (Doughty et al. ACE 2024 — `[Bloom level] + [LO] + [Keywords] + [KCs]`):
Each exercise begins with a yaml block stating its Bloom cognitive level, learning objective, keywords, and the knowledge components (KCs) it targets. This makes the cognitive demand explicit instead of buried.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Einops: Rearrange` subtopic.
You can copy the token from your Delta Drills account page.

This drill exercises the **atom `einops-rearrange`**, which bridges to the bank subtopic `Einops: Rearrange` for EWMA state. Completing all 5 exercises triggers a single `arena-rating` beacon at the end of the notebook.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "einops-rearrange"
DD_SUBTOPIC = "Einops: Rearrange"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

# Track which exercises passed in this session.
_dd_passed = set()

## einops.rearrange — quick refresher

`rearrange(tensor, pattern, **axes_lengths)` does three things with one pattern:
1. **Reorder axes** — `'h w -> w h'` is a transpose.
2. **Compose axes** — `'h w c -> (h w) c'` flattens spatial dims into one.
3. **Decompose axes** — `'(b1 b2) c -> b1 b2 c'` splits one axis into two (requires `b1=` or `b2=`).

Identifiers on the right side must match identifiers on the left — every axis is named, every axis is accounted for. No transposing semantics beyond what the pattern says.

### Exercise 3 — image flatten (axis composition)

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply axis composition `(c h w)` on the output side to flatten a 4-D tensor's trailing axes in row-major order.
> Keywords: composition, flatten, row-major
> ```

**KCs targeted:** `rearrange-axis-composition`

Implement `ex3_flatten(x)` to flatten a batch of CHW images into a batch of feature vectors.

Input shape: `(b, c, h, w)`. Output shape: `(b, c * h * w)`.

Use a **composed** axis on the right side: `(c h w)` collapses three named axes into one. Row-major order — channel varies slowest, width varies fastest.

In [ ]:
def ex3_flatten(x: Tensor) -> Tensor:
    return rearrange(x, 'b c h w -> b (c h w)')


<details><summary>Solution</summary>

```python
def ex3_flatten(x: Tensor) -> Tensor:
    return rearrange(x, 'b c h w -> b (c h w)')
```

**Why row-major?** `einops` composes axes in the order written. `(c h w)` means the stride pattern is `(c × h × w, h × w, w, 1)` — equivalent to `torch.reshape` on a contiguous tensor.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',  # single-exercise standalone — neutral signal
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()